In [ ]:
# Importaciones

import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
from scipy.stats import randint
X, y = load_breast_cancer(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

rf = RandomForestClassifier(random_state=42, n_jobs=-1)
param_dist = {
    'n_estimators': randint(100, 500),
    'max_depth': randint(3, 20),
    'min_samples_split': randint(2, 10),
    'min_samples_leaf': randint(1, 5),
    'max_features': ['sqrt', 'log2']
}
rs = RandomizedSearchCV(rf, param_distributions=param_dist, n_iter=20, random_state=42,
                        scoring='roc_auc', n_jobs=-1, cv=5)
rs.fit(X_tr, y_tr)
best = rs.best_estimator_
print('Mejores params:', rs.best_params_, '\nMejor AUC ROC CV:', rs.best_score_)


Mejores params: {'max_depth': 10, 'max_features': 'log2', 'min_samples_leaf': 2, 'min_samples_split': 7, 'n_estimators': 489} 
Mejor AUC ROC CV: 0.9910636595207502


In [2]:
# Evaluar

proba = best.predict_proba(X_te)[:,1]
print('AUC ROC test:', roc_auc_score(y_te, proba))
print('PR‑AUC test:', average_precision_score(y_te, proba))
try:
    importances = best.feature_importances_
    top_idx = np.argsort(importances)[::-1][:10]
    print('Top 10 features (idx):', top_idx)
except Exception as e:
    print('Importancias no disponibles:', e)

AUC ROC test: 0.9932914046121594
PR‑AUC test: 0.9961263862833805
Top 10 features (idx): [22 23 27 20  7  2  0  6  3 13]
